
# 02 — Raw Files Coverage Diagnostics

This notebook checks standardized raw-data coverage after:

```text
01_Make_Raw_Files_Comparable_2002.ipynb
```

## Final decisions reflected here

- Source coverage starts from **2002**, not 2000.
- The final POSet ordering set has **5 variables**, not 6.
- WGI/governance is **not** a final ordering variable.
- WGI can still be used later as contextual or sensitivity material.
- The final POSet itself is built on baseline snapshots:
  - 2007 shock → 2007 cross-section
  - 2019 shock → 2019 cross-section

## Final 5 ordering raw variables

1. `public_debt_gdp_canonical`
2. `unemployment_rate`
3. `rd_gdp`
4. `tertiary_education_25_64`
5. `energy_import_dependency_raw`

## Validation variables kept for later

- `gdp_growth`
- `unemployment_rate`
- `inflation_cpi`
- `public_debt_gdp_canonical`
- `productivity_gdp_per_hour`

## Pre-shock diagnostic windows

These are **coverage diagnostics only**, not the POSet construction rule:

- 2007 shock diagnostic window: **2002–2007**
- 2019 shock diagnostic window: **2012–2019**

The POSet itself still uses only the baseline year values.


## v2 input-path fix

This version does **not** use the old path:

```text
Data/Processed/00_Comparable_Raw_Files/Combined_Long/all_comparable_sources_long.csv
```

as the primary input.

It first looks for the clean Notebook 01 output:

```text
Data/Processed/Comparable_Raw_Files/standardized_country_year_variable_long.csv
```

Use this v2 notebook after running `01_Make_Raw_Files_Comparable_2002_v2.ipynb`.


## v3 fix

This version fixes the `NameError: df is not defined` issue by restoring the missing data-loading cell immediately after the configuration cell.


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

INPUT_DIR = PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Raw_Files_Coverage_Diagnostics"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "02_Raw_Files_Coverage_Diagnostics"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Raw_Files_Coverage_Diagnostics"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2025

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Input folder:", INPUT_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())


Run ID: 20260624_181529
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Input folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Raw_Files_Coverage_Diagnostics
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Diagnostics\02_Raw_Files_Coverage_Diagnostics


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

# Preferred input from Notebook 01 in the clean 2002 pipeline:
# Data/Processed/Comparable_Raw_Files/standardized_country_year_variable_long.csv
#
# Older paths are included only as emergency fallbacks, but the clean 2002 pipeline
# should use the first candidate.

STANDARDIZED_LONG_CANDIDATES = [
    PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files" / "standardized_country_year_variable_long.csv",
    PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files" / "standardized_all_variables_long.csv",
    PROJECT_ROOT / "Data" / "Processed" / "00_Comparable_Raw_Files" / "Combined_Long" / "all_comparable_sources_long.csv",
]

VARIABLE_METADATA_CANDIDATES = [
    PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files" / "variable_metadata.csv",
    PROJECT_ROOT / "Data" / "Processed" / "00_Comparable_Raw_Files" / "variable_metadata.csv",
]

def find_first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n" +
        "\n".join([f"- {p}" for p in candidates])
    )

STANDARDIZED_LONG_FILE = find_first_existing_path(
    STANDARDIZED_LONG_CANDIDATES,
    "standardized comparable long dataset",
)

VARIABLE_METADATA_FILE = next((p for p in VARIABLE_METADATA_CANDIDATES if p.exists()), None)
if VARIABLE_METADATA_FILE is not None:
    print("Using variable metadata:", VARIABLE_METADATA_FILE)
else:
    print("Variable metadata file not found. Continuing without it.")

FINAL_5_ORDERING_RAW_VARIABLES = [
    "public_debt_gdp_canonical",
    "unemployment_rate",
    "rd_gdp",
    "tertiary_education_25_64",
    "energy_import_dependency_raw",
]

VALIDATION_RAW_VARIABLES = [
    "gdp_growth",
    "unemployment_rate",
    "inflation_cpi",
    "public_debt_gdp_canonical",
    "productivity_gdp_per_hour",
]

BROADER_CORE_VARIABLES = sorted(set(FINAL_5_ORDERING_RAW_VARIABLES + VALIDATION_RAW_VARIABLES))

BASELINE_YEARS = [2007, 2019]

PRE_SHOCK_WINDOWS = [
    {
        "shock_id": "2007",
        "baseline_year": 2007,
        "window_start": 2002,
        "window_end": 2007,
        "note": "Pre-2007 diagnostic window starts at 2002 due to final acquisition coverage.",
    },
    {
        "shock_id": "2019",
        "baseline_year": 2019,
        "window_start": 2012,
        "window_end": 2019,
        "note": "Pre-2019 diagnostic window.",
    },
]

print("Final 5 ordering variables:")
for v in FINAL_5_ORDERING_RAW_VARIABLES:
    print("-", v)

print("\nBroader core variables for coverage/validation diagnostics:")
for v in BROADER_CORE_VARIABLES:
    print("-", v)


Using standardized comparable long dataset: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files\standardized_country_year_variable_long.csv
Using variable metadata: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files\variable_metadata.csv
Final 5 ordering variables:
- public_debt_gdp_canonical
- unemployment_rate
- rd_gdp
- tertiary_education_25_64
- energy_import_dependency_raw

Broader core variables for coverage/validation diagnostics:
- energy_import_dependency_raw
- gdp_growth
- inflation_cpi
- productivity_gdp_per_hour
- public_debt_gdp_canonical
- rd_gdp
- tertiary_education_25_64
- unemployment_rate


In [3]:

# ------------------------------------------------------
# LOAD STANDARDIZED LONG DATA
# ------------------------------------------------------

df = pd.read_csv(STANDARDIZED_LONG_FILE)

# Normalize possible old column names if an older comparable file is used.
rename_candidates = {
    "country": "Country",
    "iso3": "Country",
    "ISO3": "Country",
    "year": "Year",
    "Variable": "variable",
    "variable_name": "variable",
    "Indicator": "variable",
    "value": "Value",
    "obs_value": "Value",
    "OBS_VALUE": "Value",
}

for old_col, new_col in rename_candidates.items():
    if old_col in df.columns and new_col not in df.columns:
        df = df.rename(columns={old_col: new_col})

required_cols = {"Country", "Year", "variable", "Value"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(
        f"Standardized file missing required columns: {missing}\n"
        f"Selected file: {STANDARDIZED_LONG_FILE}\n"
        f"Available columns: {list(df.columns)}"
    )

df["Country"] = df["Country"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["variable"] = df["variable"].astype(str).str.strip()

df = (
    df.dropna(subset=["Country", "Year", "variable", "Value"])
    .assign(Year=lambda d: d["Year"].astype(int))
    .query("Year >= @START_YEAR and Year <= @END_YEAR")
    .copy()
)

# Defensive filter: only 3-letter ISO-style country codes should remain.
invalid_country_codes = df[~df["Country"].str.match(r"^[A-Z]{3}$", na=False)].copy()
invalid_country_codes.to_csv(DIAGNOSTICS_DIR / "invalid_country_codes_after_standardization.csv", index=False)

if len(invalid_country_codes) > 0:
    raise ValueError(
        "Invalid country codes found after standardization. "
        "Check invalid_country_codes_after_standardization.csv before proceeding."
    )

if VARIABLE_METADATA_FILE is not None and VARIABLE_METADATA_FILE.exists():
    variable_metadata = pd.read_csv(VARIABLE_METADATA_FILE)
else:
    variable_metadata = pd.DataFrame()

missing_final5 = sorted(set(FINAL_5_ORDERING_RAW_VARIABLES) - set(df["variable"].unique()))
if missing_final5:
    raise ValueError(
        f"Missing final 5 ordering raw variables: {missing_final5}\n"
        f"Selected input file: {STANDARDIZED_LONG_FILE}\n"
        "This usually means you are using an older comparable file. Run Notebook 01 v2 first."
    )

missing_validation = sorted(set(VALIDATION_RAW_VARIABLES) - set(df["variable"].unique()))
if missing_validation:
    raise ValueError(
        f"Missing validation raw variables: {missing_validation}\n"
        f"Selected input file: {STANDARDIZED_LONG_FILE}"
    )

print("Loaded standardized long data:", df.shape)
print("Selected input:", STANDARDIZED_LONG_FILE)
print("Countries:", df["Country"].nunique())
print("Variables:", df["variable"].nunique())
print("Years:", df["Year"].min(), "-", df["Year"].max())

display(df.head())


Loaded standardized long data: (12597, 12)
Selected input: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files\standardized_country_year_variable_long.csv
Countries: 177
Variables: 10
Years: 2002 - 2025


,Country,Year,variable,dataset_label,source_file,source_route,concept,role,Value,source_country_code,country_standardization_status,public_debt_canonical_source
0,AGO,2002,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-575.4543,AGO,kept_iso3_like,NaN
1,AGO,2003,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-520.1810,AGO,kept_iso3_like,NaN
2,AGO,2004,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-590.3533,AGO,kept_iso3_like,NaN
3,AGO,2005,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-812.8839,AGO,kept_iso3_like,NaN
4,AGO,2006,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-818.5898,AGO,kept_iso3_like,NaN


In [4]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [5]:

# ------------------------------------------------------
# OVERALL VARIABLE COVERAGE SUMMARY
# ------------------------------------------------------

variable_coverage_summary = (
    df
    .groupby("variable")
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        years_available=("Year", "nunique"),
        min_value=("Value", "min"),
        median_value=("Value", "median"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .sort_values("variable")
)

variable_coverage_summary["is_final5_ordering_raw_variable"] = variable_coverage_summary["variable"].isin(FINAL_5_ORDERING_RAW_VARIABLES)
variable_coverage_summary["is_validation_raw_variable"] = variable_coverage_summary["variable"].isin(VALIDATION_RAW_VARIABLES)
variable_coverage_summary["is_broader_core_variable"] = variable_coverage_summary["variable"].isin(BROADER_CORE_VARIABLES)

save_table(
    variable_coverage_summary,
    "variable_coverage_summary.csv",
    "Variable coverage summary",
    "Coverage summary by standardized variable.",
)

display(variable_coverage_summary)


Saved: variable_coverage_summary.csv


,variable,rows,countries,min_year,max_year,years_available,min_value,median_value,max_value,is_final5_ordering_raw_variable,is_validation_raw_variable,is_broader_core_variable
0,energy_import_dependency_raw,3038,153,2002,2023,22,-2902.1927,22.7291,450.1815,True,False,True
1,gdp_growth,1043,44,2002,2025,24,-16.0400,2.5477,24.6240,False,True,True
2,inflation_cpi,1116,48,2002,2025,24,-4.4475,2.5000,219.8845,False,True,True
3,productivity_gdp_per_hour,923,41,2002,2024,23,11.7448,51.7788,142.9044,False,True,True
4,public_debt_gdp_canonical,1662,108,2002,2025,24,-1.1707,50.2523,209.4000,True,True,True
5,public_debt_gdp_eurostat,648,27,2002,2025,24,3.9000,54.4000,209.4000,False,False,False
6,public_debt_gdp_worldbank,1060,83,2002,2024,23,-1.1707,49.9645,194.6828,False,False,False
7,rd_gdp,998,47,2002,2025,24,0.1639,1.5462,6.9589,True,False,True
8,tertiary_education_25_64,915,48,2002,2024,23,5.4441,31.3142,64.6622,True,False,True
9,unemployment_rate,1194,53,2002,2025,24,0.2510,6.5585,36.0250,True,True,True


In [6]:

# ------------------------------------------------------
# COUNTRY-LEVEL COVERAGE SUMMARY
# ------------------------------------------------------

country_coverage_summary = (
    df
    .assign(present=1)
    .pivot_table(
        index="Country",
        columns="variable",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

for var in BROADER_CORE_VARIABLES:
    if var not in country_coverage_summary.columns:
        country_coverage_summary[var] = 0

country_coverage_summary["final5_ordering_variables_present"] = country_coverage_summary[FINAL_5_ORDERING_RAW_VARIABLES].sum(axis=1)
country_coverage_summary["complete_any_year_final5_ordering"] = country_coverage_summary["final5_ordering_variables_present"].eq(len(FINAL_5_ORDERING_RAW_VARIABLES))

country_coverage_summary["validation_variables_present"] = country_coverage_summary[VALIDATION_RAW_VARIABLES].sum(axis=1)
country_coverage_summary["complete_any_year_validation_variables"] = country_coverage_summary["validation_variables_present"].eq(len(VALIDATION_RAW_VARIABLES))

country_coverage_summary["broader_core_variables_present"] = country_coverage_summary[BROADER_CORE_VARIABLES].sum(axis=1)

country_coverage_summary = country_coverage_summary.sort_values(
    ["complete_any_year_final5_ordering", "final5_ordering_variables_present", "Country"],
    ascending=[False, False, True],
).reset_index(drop=True)

save_table(
    country_coverage_summary,
    "country_coverage_summary.csv",
    "Country coverage summary",
    "Country-level variable presence summary for final 5 ordering and validation variables.",
)

display(country_coverage_summary.head(80))


Saved: country_coverage_summary.csv


variable,Country,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate,final5_ordering_variables_present,complete_any_year_final5_ordering,validation_variables_present,complete_any_year_validation_variables,broader_core_variables_present
0,AUS,1,1,1,1,1,0,1,1,1,1,5,True,5,True,8
1,AUT,1,1,1,1,1,1,0,1,1,1,5,True,5,True,8
2,BEL,1,1,1,1,1,1,0,1,1,1,5,True,5,True,8
3,BGR,1,0,1,1,1,1,0,1,1,1,5,True,4,False,7
4,CAN,1,1,1,1,1,0,1,1,1,1,5,True,5,True,8
5,CHE,1,1,1,1,1,0,1,1,1,1,5,True,5,True,8
6,COL,1,1,1,1,1,0,1,1,1,1,5,True,5,True,8
7,CZE,1,1,1,1,1,1,0,1,1,1,5,True,5,True,8
8,DEU,1,1,1,1,1,1,0,1,1,1,5,True,5,True,8
9,DNK,1,1,1,1,1,1,0,1,1,1,5,True,5,True,8


In [7]:

# ------------------------------------------------------
# COUNTRY-YEAR COVERAGE SUMMARY
# ------------------------------------------------------

country_year_coverage_summary = (
    df[df["variable"].isin(BROADER_CORE_VARIABLES)]
    .assign(present=1)
    .pivot_table(
        index=["Country", "Year"],
        columns="variable",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

for var in BROADER_CORE_VARIABLES:
    if var not in country_year_coverage_summary.columns:
        country_year_coverage_summary[var] = 0

country_year_coverage_summary["final5_ordering_variables_present"] = country_year_coverage_summary[FINAL_5_ORDERING_RAW_VARIABLES].sum(axis=1)
country_year_coverage_summary["complete_final5_ordering_country_year"] = country_year_coverage_summary["final5_ordering_variables_present"].eq(len(FINAL_5_ORDERING_RAW_VARIABLES))

country_year_coverage_summary["validation_variables_present"] = country_year_coverage_summary[VALIDATION_RAW_VARIABLES].sum(axis=1)
country_year_coverage_summary["complete_validation_country_year"] = country_year_coverage_summary["validation_variables_present"].eq(len(VALIDATION_RAW_VARIABLES))

country_year_coverage_summary["broader_core_variables_present"] = country_year_coverage_summary[BROADER_CORE_VARIABLES].sum(axis=1)

save_table(
    country_year_coverage_summary,
    "country_year_coverage_summary.csv",
    "Country-year coverage summary",
    "Country-year coverage across final 5 ordering variables and validation variables.",
)

display(country_year_coverage_summary.head())


Saved: country_year_coverage_summary.csv


variable,Country,Year,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,rd_gdp,tertiary_education_25_64,unemployment_rate,final5_ordering_variables_present,complete_final5_ordering_country_year,validation_variables_present,complete_validation_country_year,broader_core_variables_present
0,AGO,2002,1,0,0,0,0,0,0,0,1,False,0,False,1
1,AGO,2003,1,0,0,0,0,0,0,0,1,False,0,False,1
2,AGO,2004,1,0,0,0,0,0,0,0,1,False,0,False,1
3,AGO,2005,1,0,0,0,0,0,0,0,1,False,0,False,1
4,AGO,2006,1,0,0,0,0,0,0,0,1,False,0,False,1


In [8]:

# ------------------------------------------------------
# BASELINE YEAR COVERAGE FOR FINAL 5 ORDERING VARIABLES
# ------------------------------------------------------

baseline_rows = []

for baseline_year in BASELINE_YEARS:
    subset = df[
        df["Year"].eq(baseline_year)
        & df["variable"].isin(FINAL_5_ORDERING_RAW_VARIABLES)
    ].copy()

    presence = (
        subset
        .assign(present=1)
        .pivot_table(
            index="Country",
            columns="variable",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
    )

    for var in FINAL_5_ORDERING_RAW_VARIABLES:
        if var not in presence.columns:
            presence[var] = 0

    presence = presence[FINAL_5_ORDERING_RAW_VARIABLES]
    presence["baseline_year"] = baseline_year
    presence["final5_variables_present"] = presence[FINAL_5_ORDERING_RAW_VARIABLES].sum(axis=1)
    presence["final5_variables_missing"] = len(FINAL_5_ORDERING_RAW_VARIABLES) - presence["final5_variables_present"]
    presence["complete_case_final5_baseline"] = presence["final5_variables_present"].eq(len(FINAL_5_ORDERING_RAW_VARIABLES))
    baseline_rows.append(presence.reset_index())

baseline_year_coverage_final5 = pd.concat(baseline_rows, ignore_index=True)

baseline_year_coverage_final5 = baseline_year_coverage_final5[
    [
        "Country",
        "baseline_year",
        "complete_case_final5_baseline",
        "final5_variables_present",
        "final5_variables_missing",
    ] + FINAL_5_ORDERING_RAW_VARIABLES
].sort_values(
    ["baseline_year", "complete_case_final5_baseline", "final5_variables_present", "Country"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

baseline_year_coverage_final5_summary = (
    baseline_year_coverage_final5
    .groupby("baseline_year")
    .agg(
        countries_checked=("Country", "nunique"),
        complete_case_countries=("complete_case_final5_baseline", "sum"),
        mean_final5_variables_present=("final5_variables_present", "mean"),
        median_final5_variables_present=("final5_variables_present", "median"),
    )
    .reset_index()
)

save_table(
    baseline_year_coverage_final5,
    "baseline_year_coverage_final5_2007_2019.csv",
    "Final 5 baseline-year coverage",
    "Country-level complete-case check for final 5 ordering variables in 2007 and 2019.",
)

save_table(
    baseline_year_coverage_final5_summary,
    "baseline_year_coverage_final5_summary.csv",
    "Final 5 baseline-year coverage summary",
    "Summary of complete countries for final 5 ordering variables in 2007 and 2019.",
)

display(baseline_year_coverage_final5_summary)
display(baseline_year_coverage_final5.head(100))


Saved: baseline_year_coverage_final5_2007_2019.csv
Saved: baseline_year_coverage_final5_summary.csv


,baseline_year,countries_checked,complete_case_countries,mean_final5_variables_present,median_final5_variables_present
0,2007,152,25,2.1974,1.0000
1,2019,161,35,2.2857,1.0000


variable,Country,baseline_year,complete_case_final5_baseline,final5_variables_present,final5_variables_missing,public_debt_gdp_canonical,unemployment_rate,rd_gdp,tertiary_education_25_64,energy_import_dependency_raw
0,AUT,2007,True,5,0,1,1,1,1,1
1,BEL,2007,True,5,0,1,1,1,1,1
2,CAN,2007,True,5,0,1,1,1,1,1
3,CZE,2007,True,5,0,1,1,1,1,1
4,DEU,2007,True,5,0,1,1,1,1,1
5,DNK,2007,True,5,0,1,1,1,1,1
6,ESP,2007,True,5,0,1,1,1,1,1
7,EST,2007,True,5,0,1,1,1,1,1
8,FIN,2007,True,5,0,1,1,1,1,1
9,FRA,2007,True,5,0,1,1,1,1,1


In [9]:

# ------------------------------------------------------
# BROADER BASELINE YEAR COVERAGE FOR VALIDATION VARIABLES
# ------------------------------------------------------

validation_baseline_rows = []

for baseline_year in BASELINE_YEARS:
    subset = df[
        df["Year"].eq(baseline_year)
        & df["variable"].isin(VALIDATION_RAW_VARIABLES)
    ].copy()

    presence = (
        subset
        .assign(present=1)
        .pivot_table(
            index="Country",
            columns="variable",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
    )

    for var in VALIDATION_RAW_VARIABLES:
        if var not in presence.columns:
            presence[var] = 0

    presence = presence[VALIDATION_RAW_VARIABLES]
    presence["baseline_year"] = baseline_year
    presence["validation_variables_present"] = presence[VALIDATION_RAW_VARIABLES].sum(axis=1)
    presence["complete_case_validation_baseline"] = presence["validation_variables_present"].eq(len(VALIDATION_RAW_VARIABLES))
    validation_baseline_rows.append(presence.reset_index())

baseline_year_coverage_validation = pd.concat(validation_baseline_rows, ignore_index=True)

baseline_year_coverage_validation_summary = (
    baseline_year_coverage_validation
    .groupby("baseline_year")
    .agg(
        countries_checked=("Country", "nunique"),
        complete_case_countries=("complete_case_validation_baseline", "sum"),
        mean_validation_variables_present=("validation_variables_present", "mean"),
    )
    .reset_index()
)

save_table(
    baseline_year_coverage_validation,
    "baseline_year_coverage_validation_2007_2019.csv",
    "Validation baseline-year coverage",
    "Country-level check for validation variable availability in 2007 and 2019.",
)

save_table(
    baseline_year_coverage_validation_summary,
    "baseline_year_coverage_validation_summary.csv",
    "Validation baseline-year coverage summary",
    "Summary of validation variable availability in 2007 and 2019.",
)

display(baseline_year_coverage_validation_summary)


Saved: baseline_year_coverage_validation_2007_2019.csv
Saved: baseline_year_coverage_validation_summary.csv


,baseline_year,countries_checked,complete_case_countries,mean_validation_variables_present
0,2007,83,29,2.9880
1,2019,93,33,2.8710


In [10]:

# ------------------------------------------------------
# PRE-SHOCK WINDOW COVERAGE DIAGNOSTICS
# ------------------------------------------------------
# Diagnostic only. The final POSet uses baseline snapshots, not averages.

pre_shock_window_rows = []

for window in PRE_SHOCK_WINDOWS:
    shock_id = window["shock_id"]
    start = window["window_start"]
    end = window["window_end"]
    expected_years = list(range(start, end + 1))
    expected_year_count = len(expected_years)

    subset = df[
        (df["Year"] >= start)
        & (df["Year"] <= end)
        & (df["variable"].isin(BROADER_CORE_VARIABLES))
    ].copy()

    coverage = (
        subset
        .groupby(["Country", "variable"])
        .agg(
            years_available=("Year", "nunique"),
            min_year=("Year", "min"),
            max_year=("Year", "max"),
        )
        .reset_index()
    )

    coverage["shock_id"] = shock_id
    coverage["window_start"] = start
    coverage["window_end"] = end
    coverage["expected_year_count"] = expected_year_count
    coverage["coverage_rate_in_window"] = coverage["years_available"] / expected_year_count
    pre_shock_window_rows.append(coverage)

pre_shock_window_coverage = pd.concat(pre_shock_window_rows, ignore_index=True)

pre_shock_window_coverage_summary = (
    pre_shock_window_coverage
    .groupby(["shock_id", "variable"])
    .agg(
        countries_with_any_data=("Country", "nunique"),
        mean_years_available=("years_available", "mean"),
        median_years_available=("years_available", "median"),
        mean_coverage_rate=("coverage_rate_in_window", "mean"),
    )
    .reset_index()
    .sort_values(["shock_id", "variable"])
)

save_table(
    pre_shock_window_coverage,
    "pre_shock_window_coverage.csv",
    "Pre-shock window coverage",
    "Coverage by country-variable in 2002–2007 and 2012–2019 diagnostic windows.",
)

save_table(
    pre_shock_window_coverage_summary,
    "pre_shock_window_coverage_summary.csv",
    "Pre-shock window coverage summary",
    "Variable-level coverage summaries for diagnostic pre-shock windows.",
)

display(pre_shock_window_coverage_summary)


Saved: pre_shock_window_coverage.csv
Saved: pre_shock_window_coverage_summary.csv


,shock_id,variable,countries_with_any_data,mean_years_available,median_years_available,mean_coverage_rate
0,2007,energy_import_dependency_raw,141,5.7943,6.0000,0.9657
1,2007,gdp_growth,44,6.0000,6.0000,1.0000
2,2007,inflation_cpi,47,6.0000,6.0000,1.0000
3,2007,productivity_gdp_per_hour,40,6.0000,6.0000,1.0000
4,2007,public_debt_gdp_canonical,72,5.2500,6.0000,0.8750
5,2007,rd_gdp,46,5.5435,6.0000,0.9239
6,2007,tertiary_education_25_64,38,5.3421,6.0000,0.8904
7,2007,unemployment_rate,50,5.7600,6.0000,0.9600
8,2019,energy_import_dependency_raw,150,7.7000,8.0000,0.9625
9,2019,gdp_growth,44,8.0000,8.0000,1.0000


In [11]:

# ------------------------------------------------------
# RECOMMENDED COUNTRY KEEP / REVIEW / DROP LIST
# ------------------------------------------------------
# This is a recommendation based on coverage only.
# Final sample choice is still made later in the master dataset / Pre-POSet EDA.

baseline_pivot = (
    baseline_year_coverage_final5
    .pivot_table(
        index="Country",
        columns="baseline_year",
        values="complete_case_final5_baseline",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)

for year in BASELINE_YEARS:
    if year not in baseline_pivot.columns:
        baseline_pivot[year] = False

baseline_pivot = baseline_pivot.rename(columns={
    2007: "complete_final5_2007",
    2019: "complete_final5_2019",
})

recommended_country_keep_drop_review_list = baseline_pivot.copy()

recommended_country_keep_drop_review_list["coverage_decision_recommendation"] = np.select(
    [
        recommended_country_keep_drop_review_list["complete_final5_2007"] & recommended_country_keep_drop_review_list["complete_final5_2019"],
        recommended_country_keep_drop_review_list["complete_final5_2007"] | recommended_country_keep_drop_review_list["complete_final5_2019"],
    ],
    [
        "keep_for_both_shocks",
        "review_or_keep_for_one_shock_only",
    ],
    default="drop_or_exclude_from_main_poset",
)

recommended_country_keep_drop_review_list["reason"] = np.select(
    [
        recommended_country_keep_drop_review_list["coverage_decision_recommendation"].eq("keep_for_both_shocks"),
        recommended_country_keep_drop_review_list["coverage_decision_recommendation"].eq("review_or_keep_for_one_shock_only"),
    ],
    [
        "Complete final 5 ordering-variable coverage in both 2007 and 2019.",
        "Complete final 5 ordering-variable coverage in only one baseline year.",
    ],
    default="Incomplete final 5 ordering-variable coverage in both baseline years.",
)

recommended_country_keep_drop_review_list = recommended_country_keep_drop_review_list.sort_values(
    ["coverage_decision_recommendation", "Country"]
).reset_index(drop=True)

save_table(
    recommended_country_keep_drop_review_list,
    "recommended_country_keep_drop_review_list.csv",
    "Recommended country keep/drop/review list",
    "Coverage-based recommendation for final 5-variable POSet baseline years.",
)

decision_summary = (
    recommended_country_keep_drop_review_list
    .groupby("coverage_decision_recommendation")
    .agg(countries=("Country", "nunique"))
    .reset_index()
)

save_table(
    decision_summary,
    "recommended_country_decision_summary.csv",
    "Recommended country decision summary",
    "Counts by coverage-based country recommendation category.",
)

display(decision_summary)
display(recommended_country_keep_drop_review_list.head(100))


Saved: recommended_country_keep_drop_review_list.csv
Saved: recommended_country_decision_summary.csv


,coverage_decision_recommendation,countries
0,drop_or_exclude_from_main_poset,134
1,keep_for_both_shocks,25
2,review_or_keep_for_one_shock_only,10


baseline_year,Country,complete_final5_2007,complete_final5_2019,coverage_decision_recommendation,reason
0,AGO,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
1,ALB,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
2,AND,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
3,ARB,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
4,ARE,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
5,ARG,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
6,ARM,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
7,AZE,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
8,BEN,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...
9,BFA,False,False,drop_or_exclude_from_main_poset,Incomplete final 5 ordering-variable coverage ...


In [12]:

# ------------------------------------------------------
# VALUE-RANGE SANITY CHECKS
# ------------------------------------------------------

def variable_range_note(row):
    var = row["variable"]
    min_value = row["min_value"]
    max_value = row["max_value"]

    if var == "energy_import_dependency_raw":
        return "Raw energy dependency can be negative for net exporters and above 100 for high import dependence. This is expected before energy-security transformation."
    if var == "public_debt_gdp_canonical" and min_value < 0:
        return "Negative debt value detected in raw canonical debt. Check whether affected countries enter final baseline samples."
    if var == "unemployment_rate" and min_value < 0:
        return "Unemployment below zero is unexpected."
    if var == "tertiary_education_25_64" and (min_value < 0 or max_value > 100):
        return "Tertiary attainment outside 0–100 range."
    if var == "rd_gdp" and min_value < 0:
        return "R&D/GDP below zero is unexpected."
    return "OK"

value_range_sanity_checks = variable_coverage_summary.copy()
value_range_sanity_checks["range_note"] = value_range_sanity_checks.apply(variable_range_note, axis=1)

save_table(
    value_range_sanity_checks,
    "value_range_sanity_checks.csv",
    "Value-range sanity checks",
    "Range checks for standardized variables.",
)

display(value_range_sanity_checks[["variable", "min_value", "max_value", "range_note"]])


Saved: value_range_sanity_checks.csv


,variable,min_value,max_value,range_note
0,energy_import_dependency_raw,-2902.1927,450.1815,Raw energy dependency can be negative for net ...
1,gdp_growth,-16.0400,24.6240,OK
2,inflation_cpi,-4.4475,219.8845,OK
3,productivity_gdp_per_hour,11.7448,142.9044,OK
4,public_debt_gdp_canonical,-1.1707,209.4000,Negative debt value detected in raw canonical ...
5,public_debt_gdp_eurostat,3.9000,209.4000,OK
6,public_debt_gdp_worldbank,-1.1707,194.6828,OK
7,rd_gdp,0.1639,6.9589,OK
8,tertiary_education_25_64,5.4441,64.6622,OK
9,unemployment_rate,0.2510,36.0250,OK


In [13]:

# ------------------------------------------------------
# OVERALL COVERAGE SUMMARY
# ------------------------------------------------------

coverage_summary_overall = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "standardized_rows": len(df),
    "countries_total": df["Country"].nunique(),
    "variables_total": df["variable"].nunique(),
    "final5_ordering_variable_count": len(FINAL_5_ORDERING_RAW_VARIABLES),
    "validation_variable_count": len(VALIDATION_RAW_VARIABLES),
    "complete_final5_baseline_countries_2007": int(
        baseline_year_coverage_final5_summary.loc[
            baseline_year_coverage_final5_summary["baseline_year"].eq(2007),
            "complete_case_countries"
        ].iloc[0]
    ),
    "complete_final5_baseline_countries_2019": int(
        baseline_year_coverage_final5_summary.loc[
            baseline_year_coverage_final5_summary["baseline_year"].eq(2019),
            "complete_case_countries"
        ].iloc[0]
    ),
    "wgi_role": "contextual_or_sensitivity_only_not_final_ordering",
    "poset_baseline_years": "2007,2019",
    "pre_shock_diagnostic_windows": "2002-2007;2012-2019",
}])

save_table(
    coverage_summary_overall,
    "coverage_summary_overall.csv",
    "Overall coverage summary",
    "Top-level summary for raw-data coverage diagnostics.",
)

display(coverage_summary_overall)


Saved: coverage_summary_overall.csv


,run_id,run_timestamp,start_year,end_year,standardized_rows,countries_total,variables_total,final5_ordering_variable_count,validation_variable_count,complete_final5_baseline_countries_2007,complete_final5_baseline_countries_2019,wgi_role,poset_baseline_years,pre_shock_diagnostic_windows
0,20260624_181529,2026-06-24 18:15:29,2002,2025,12597,177,10,5,5,25,35,contextual_or_sensitivity_only_not_final_ordering,"2007,2019",2002-2007;2012-2019


In [14]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "02_Raw_Files_Coverage_Diagnostics_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    coverage_summary_overall.to_excel(writer, sheet_name="overall_summary", index=False)
    variable_coverage_summary.to_excel(writer, sheet_name="variable_coverage", index=False)
    country_coverage_summary.to_excel(writer, sheet_name="country_coverage", index=False)
    country_year_coverage_summary.to_excel(writer, sheet_name="country_year_coverage", index=False)
    baseline_year_coverage_final5_summary.to_excel(writer, sheet_name="final5_baseline_summary", index=False)
    baseline_year_coverage_final5.to_excel(writer, sheet_name="final5_baseline_detail", index=False)
    baseline_year_coverage_validation_summary.to_excel(writer, sheet_name="validation_baseline_summary", index=False)
    baseline_year_coverage_validation.to_excel(writer, sheet_name="validation_baseline_detail", index=False)
    pre_shock_window_coverage_summary.to_excel(writer, sheet_name="pre_shock_summary", index=False)
    pre_shock_window_coverage.to_excel(writer, sheet_name="pre_shock_detail", index=False)
    recommended_country_keep_drop_review_list.to_excel(writer, sheet_name="country_recommendation", index=False)
    value_range_sanity_checks.to_excel(writer, sheet_name="value_ranges", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\02_Raw_Files_Coverage_Diagnostics_Audit.xlsx


In [15]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "coverage_diagnostics_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "coverage_diagnostics_table_inventory.csv", index=False)

expected_outputs = [
    "coverage_summary_overall.csv",
    "variable_coverage_summary.csv",
    "country_coverage_summary.csv",
    "country_year_coverage_summary.csv",
    "baseline_year_coverage_final5_2007_2019.csv",
    "baseline_year_coverage_final5_summary.csv",
    "baseline_year_coverage_validation_2007_2019.csv",
    "baseline_year_coverage_validation_summary.csv",
    "pre_shock_window_coverage.csv",
    "pre_shock_window_coverage_summary.csv",
    "recommended_country_keep_drop_review_list.csv",
    "recommended_country_decision_summary.csv",
    "value_range_sanity_checks.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("02 RAW FILES COVERAGE DIAGNOSTICS COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- coverage_summary_overall.csv")
print("- baseline_year_coverage_final5_summary.csv")
print("- recommended_country_keep_drop_review_list.csv")
print("- value_range_sanity_checks.csv")

print("\nNext notebook:")
print("03_GDP_Recovery_Dynamic_Baseline_2002.ipynb")


02 RAW FILES COVERAGE DIAGNOSTICS COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,coverage_summary_overall.csv,True,True,True
1,variable_coverage_summary.csv,True,True,True
2,country_coverage_summary.csv,True,True,True
3,country_year_coverage_summary.csv,True,True,True
4,baseline_year_coverage_final5_2007_2019.csv,True,True,True
5,baseline_year_coverage_final5_summary.csv,True,True,True
6,baseline_year_coverage_validation_2007_2019.csv,True,True,True
7,baseline_year_coverage_validation_summary.csv,True,True,True
8,pre_shock_window_coverage.csv,True,True,True
9,pre_shock_window_coverage_summary.csv,True,True,True



Key outputs to inspect/send:
- coverage_summary_overall.csv
- baseline_year_coverage_final5_summary.csv
- recommended_country_keep_drop_review_list.csv
- value_range_sanity_checks.csv

Next notebook:
03_GDP_Recovery_Dynamic_Baseline_2002.ipynb
